# CIDS · 16 — S7 oracle: how much can a boundary refiner POSSIBLY gain? (no training)

The refiner may only change voxels within R mm of the jury mask's edge (the **editable band**). Here every editable voxel is set to the ground truth — a cheating **upper limit** that no real refiner can beat:

`oracle(R) = jury mask outside the band  +  ground truth inside the band`

Reported for **R = 4, 8, 12 mm** on **val** (decision split) and **train** (the refiner's training masks):
Dice of the jury mask vs the oracle, NSD / HD95 at 8 mm, fixable vs **unreachable** errors (ml), by collection and tumour size.

**Decision:** small 8 mm gain on val → a boundary refiner cannot help much (stop, report it). Large gain → train (arms A / B).
Ground truth is used for this analysis only, never as a model input.

**Settings:** **no accelerator needed (CPU)** · Internet ON. **Inputs:** cache · notebook 12 output (or dataset `cids-s3f-jury-masks`). **Time:** ~15–25 min.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
S3F_DIR    = '/kaggle/input'   # notebook 12 output is found automatically (folder with summary.json + masks/)
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/s7_oracle'
PATHS = [CACHE_ROOT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
from pathlib import Path
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
hits = [p.parent for p in Path(S3F_DIR).rglob('summary.json') if (p.parent / 'masks').is_dir() or (p.parent / 'masks.zip').exists()]
print('notebook 12 output:', hits)
assert hits, 'attach the notebook 12 output (Add Input -> Your Work -> Notebooks -> 12_s3f_jury_masks)'
S3F_DIR = str(hits[0])

## Quick check (~1 min): 5 val cases
Check: a VAL table with three columns (4 / 8 / 12 mm). Numbers on 5 cases mean nothing.

In [ ]:
sh(f'python scripts/s7_oracle.py --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --out /kaggle/working/oracle_smoke --limit 5')
sh(f'rm -rf /kaggle/working/oracle_smoke')

## Full report (~15–25 min): val (120) + train (1080)
**Send back:** the printed VAL and TRAIN blocks, and `oracle_summary.json`.

In [ ]:
sh(f'python scripts/s7_oracle.py --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --out {OUT} --split val,train')

In [ ]:
import json
print(json.dumps(json.load(open(f'{OUT}/oracle_summary.json')), indent=1)[:6000])